# 01 - Làm sạch dữ liệu & Phân tích khám phá (EDA)
**Mục tiêu:** Load 11 sheet, enrichment null, chuẩn hóa màu, EDA.  
**Output:** `fact_sales_clean.parquet` + dimension tables.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

DATA_RAW = Path('../data/Data Explorers 2026 - Final Dataset .xlsx')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
(DATA_PROCESSED / 'forecasts').mkdir(exist_ok=True)

EXCEL_EPOCH = pd.Timestamp('1899-12-30')

def convert_excel_date(series):
    def _conv(v):
        if pd.isna(v):
            return pd.NaT
        try:
            return EXCEL_EPOCH + pd.Timedelta(days=float(v))
        except Exception:
            try:
                return pd.Timestamp(v)
            except Exception:
                return pd.NaT
    return series.apply(_conv)

def safe_str(val):
    if pd.isna(val):
        return np.nan
    return str(val).strip()

print('Setup complete.')

Setup complete.


## 1.1 Load tất cả các sheet

In [2]:
sheet_names = [
    'fact_sales', 'customer', 'email_log', 'order_line',
    'product_group', 'product_line', 'product_price',
    'province', 'product', 'sales_order', 'code lạ'
]

dfs = {}
for name in sheet_names:
    dfs[name] = pd.read_excel(DATA_RAW, sheet_name=name)
    print(f'{name}: {dfs[name].shape}')

fact_sales = dfs['fact_sales'].copy()
customer = dfs['customer'].copy()
email_log = dfs['email_log'].copy()
order_line = dfs['order_line'].copy()
product_group = dfs['product_group'].copy()
product_line = dfs['product_line'].copy()
product_price = dfs['product_price'].copy()
province = dfs['province'].copy()
product = dfs['product'].copy()
sales_order = dfs['sales_order'].copy()
code_la = dfs['code lạ'].copy()

fact_sales: (25739, 24)


customer: (798, 7)


email_log: (1132, 8)


order_line: (25754, 7)


product_group: (5, 3)


...

product_price: (1016, 4)


province: (75, 3)


product: (247, 6)


sales_order: (2759, 12)


code lạ: (17, 6)


## 1.2 Chuẩn hóa kiểu dữ liệu & chuyển ngày

In [3]:
for t in [fact_sales, order_line, product, product_price, code_la]:
    if 'product_code' in t.columns:
        t['product_code'] = t['product_code'].apply(safe_str)

for t in [fact_sales, customer, sales_order]:
    if 'customer_code' in t.columns:
        t['customer_code'] = t['customer_code'].apply(safe_str)

for t in [product_group, product_line]:
    if 'group_code' in t.columns:
        t['group_code'] = t['group_code'].apply(safe_str)

for t in [customer, province, product_line, fact_sales, product]:
    if 'province_id' in t.columns:
        t['province_id'] = pd.to_numeric(t['province_id'], errors='coerce')
    if 'line_id' in t.columns:
        t['line_id'] = pd.to_numeric(t['line_id'], errors='coerce')

if 'line_id_fk' in fact_sales.columns:
    fact_sales['line_id_fk'] = pd.to_numeric(fact_sales['line_id_fk'], errors='coerce')

for t in [fact_sales, product]:
    if 'color' in t.columns:
        t['color'] = t['color'].apply(safe_str)

fact_sales['order_date'] = convert_excel_date(fact_sales['order_date'])
sales_order['order_date'] = convert_excel_date(sales_order['order_date'])
product_price['effective_from'] = convert_excel_date(product_price['effective_from'])

print('Date range fact_sales:', fact_sales['order_date'].min().date(), '->', fact_sales['order_date'].max().date())
print('Date range sales_order:', sales_order['order_date'].min().date(), '->', sales_order['order_date'].max().date())
print('Types standardized.')

Date range fact_sales: 2025-01-02 -> 2026-03-31
Date range sales_order: 2025-01-02 -> 2026-03-31
Types standardized.


## 1.3 Loại bỏ code lạ

In [4]:
bad_codes = set(code_la['product_code'].dropna().unique())
print(f'Số mã code lạ: {len(bad_codes)}')

b1 = len(fact_sales)
fact_sales = fact_sales[~fact_sales['product_code'].isin(bad_codes)].copy()
print(f'fact_sales: {b1} -> {len(fact_sales)} (loại {b1 - len(fact_sales)})')

b2 = len(order_line)
order_line = order_line[~order_line['product_code'].isin(bad_codes)].copy()
print(f'order_line: {b2} -> {len(order_line)} (loại {b2 - len(order_line)})')

Số mã code lạ: 17
fact_sales: 25739 -> 25660 (loại 79)
order_line: 25754 -> 25674 (loại 80)


## 1.4 Enrichment - Xử lý null

In [5]:
print('Null TRƯỚC enrichment:')
nc = fact_sales.isnull().sum()
print(nc[nc > 0])

Null TRƯỚC enrichment:
province_id      1244
province_name    1244
region           1244
color               1
line_id_fk       5285
line_name        5285
group_code       5285
group_name       5285
dtype: int64


In [6]:
product_enriched = product.merge(
    product_line[['line_id', 'line_name', 'group_code']], on='line_id', how='left'
).merge(
    product_group[['group_code', 'group_name']], on='group_code', how='left'
)

prod_lkp = product_enriched[['product_code','line_id','line_name','group_code','group_name','color']].copy()
prod_lkp.columns = ['product_code','lkp_lid','lkp_lname','lkp_gcode','lkp_gname','lkp_color']

cust_lkp = customer[['customer_code','province_id']].merge(
    province[['province_id','province_name','region']], on='province_id', how='left'
)
cust_lkp.columns = ['customer_code','lkp_pid','lkp_pname','lkp_region']

fs = fact_sales.merge(prod_lkp, on='product_code', how='left')
fs = fs.merge(cust_lkp, on='customer_code', how='left')

fs['line_id_fk']    = fs['line_id_fk'].fillna(fs['lkp_lid'])
fs['line_name']     = fs['line_name'].fillna(fs['lkp_lname'])
fs['group_code']    = fs['group_code'].fillna(fs['lkp_gcode'])
fs['group_name']    = fs['group_name'].fillna(fs['lkp_gname'])
fs['color']         = fs['color'].fillna(fs['lkp_color'])
fs['province_id']   = fs['province_id'].fillna(fs['lkp_pid'])
fs['province_name'] = fs['province_name'].fillna(fs['lkp_pname'])
fs['region']        = fs['region'].fillna(fs['lkp_region'])

keep = [
    'fact_id','order_date','fiscal_year','fiscal_quarter','fiscal_month',
    'week_of_year','so_number','order_id','line_id','customer_code',
    'customer_name','province_id','province_name','region',
    'product_code','product_name','color','line_id_fk','line_name',
    'group_code','group_name','quantity','unit_price','line_total'
]
fact_sales_clean = fs[[c for c in keep if c in fs.columns]].copy()

print('Null SAU enrichment:')
nc2 = fact_sales_clean.isnull().sum()
print(nc2[nc2 > 0])
print(f'Tổng dòng: {len(fact_sales_clean)}')

Null SAU enrichment:
province_id      1244
province_name    1244
region           1244
color               1
line_id_fk       5285
line_name        5285
group_code       5285
group_name       5285
dtype: int64
Tổng dòng: 25660


## 1.5 Chuẩn hóa màu sắc

In [7]:
def std_color(c):
    if pd.isna(c) or str(c).strip().lower() in ('nan','nat',''):
        return 'không_rõ'
    return str(c).strip().lower()

fact_sales_clean['color_std'] = fact_sales_clean['color'].apply(std_color)
print(f'Unique colors raw: {fact_sales_clean["color"].nunique()}')
print(f'Unique colors std: {fact_sales_clean["color_std"].nunique()}')
print(fact_sales_clean['color_std'].value_counts().head(15))

Unique colors raw: 51
Unique colors std: 44
color_std
đen            3883
kem            3573
ghi            2899
hồng           2346
trắng          2135
xanh           1731
café/nâu       1427
xanh dương     1179
xanh mint       807
đỏ              603
coban           479
đỏ tươi         440
cam             417
pastel xanh     417
đỏ đun          384
Name: count, dtype: int64


## 1.6 Kiểm tra chất lượng

In [8]:
print('=== fact_id unique ===')
print(f'  Total: {len(fact_sales_clean)}, Unique: {fact_sales_clean["fact_id"].nunique()}')

print('\n=== Fiscal alignment ===')
mc = (fact_sales_clean['fiscal_month'] != fact_sales_clean['order_date'].dt.month).sum()
yc = (fact_sales_clean['fiscal_year'] != fact_sales_clean['order_date'].dt.year).sum()
print(f'  Month mismatch: {mc}, Year mismatch: {yc}')

print('\n=== Email log ===')
rev_so = email_log[email_log['processing_status']=='REVIEW_REQUIRED']['so_number']
valid_so = set(sales_order['so_number'])
print(f'  REVIEW_REQUIRED: {len(rev_so)}, valid: {rev_so.isin(valid_so).sum()}')

print('\n=== Summary ===')
print(f'  Period: {fact_sales_clean["order_date"].min().date()} -> {fact_sales_clean["order_date"].max().date()}')
print(f'  Revenue: {fact_sales_clean["line_total"].sum():,.0f} VND')
print(f'  Quantity: {fact_sales_clean["quantity"].sum():,.0f}')
print(f'  Orders: {fact_sales_clean["so_number"].nunique()}')
print(f'  Dealers: {fact_sales_clean["customer_code"].nunique()}')
print(f'  SKUs: {fact_sales_clean["product_code"].nunique()}')
print(f'  Groups: {fact_sales_clean["group_code"].dropna().nunique()}')

=== fact_id unique ===
  Total: 25660, Unique: 24660

=== Fiscal alignment ===
  Month mismatch: 0, Year mismatch: 0

=== Email log ===
  REVIEW_REQUIRED: 282, valid: 282

=== Summary ===
  Period: 2025-01-02 -> 2026-03-31
  Revenue: 108,435,603,152 VND
  Quantity: 71,378
  Orders: 2629
  Dealers: 787
  SKUs: 246
  Groups: 5


## 1.7 EDA

In [9]:
wrev = fact_sales_clean.set_index('order_date').resample('W')['line_total'].sum().reset_index()
wrev.columns = ['week','revenue']
fig = px.line(wrev, x='week', y='revenue', title='Doanh thu theo tuần')
fig.show()

In [10]:
tmp = fact_sales_clean.copy()
tmp['ym'] = tmp['order_date'].dt.to_period('M').astype(str)
gm = tmp.groupby(['ym','group_code'])['line_total'].sum().reset_index()
fig = px.bar(gm, x='ym', y='line_total', color='group_code', barmode='stack',
             title='Doanh thu theo tháng & nhóm SP',
             labels={'ym':'Tháng','line_total':'Doanh thu','group_code':'Nhóm'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [11]:
rr = fact_sales_clean.groupby('region').agg(rev=('line_total','sum'),orders=('so_number','nunique')).reset_index().sort_values('rev',ascending=False)
fig = px.bar(rr, x='region', y='rev', title='Doanh thu theo vùng', text_auto='.2s')
fig.show()

In [12]:
os = fact_sales_clean.groupby('so_number')['line_total'].sum().reset_index()
fig = px.histogram(os, x='line_total', nbins=50, title='Phân bổ giá trị đơn hàng')
fig.show()

In [13]:
from statsmodels.tsa.seasonal import STL

wts = fact_sales_clean.set_index('order_date').resample('W')['line_total'].sum()
wts = wts[wts.index >= '2025-01-01']
stl = STL(wts, period=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
stl.observed.plot(ax=axes[0], title='Observed')
stl.trend.plot(ax=axes[1], title='Trend')
stl.seasonal.plot(ax=axes[2], title='Seasonal')
stl.resid.plot(ax=axes[3], title='Residual')
plt.suptitle('STL Decomposition', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/stl_decomposition.png', dpi=100, bbox_inches='tight')
plt.close()
print('STL saved.')

STL saved.


In [14]:
t20 = fact_sales_clean.groupby(['customer_code','customer_name']).agg(rev=('line_total','sum')).reset_index().nlargest(20,'rev')
fig = px.bar(t20, x='rev', y='customer_name', orientation='h', title='Top 20 đại lý')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=600)
fig.show()

In [15]:
tmp2 = fact_sales_clean.copy()
tmp2['ym'] = tmp2['order_date'].dt.to_period('M').astype(str)
top_c = fact_sales_clean['color_std'].value_counts().head(12).index.tolist()
cm = tmp2[tmp2['color_std'].isin(top_c)]
cp = cm.groupby(['ym','color_std'])['quantity'].sum().reset_index()
cpw = cp.pivot(index='ym', columns='color_std', values='quantity').fillna(0)
fig = px.imshow(cpw.T, aspect='auto', title='Heatmap: Màu sắc x Tháng (Top 12)')
fig.update_layout(height=500)
fig.show()

## 1.8 Lưu dữ liệu

In [16]:
for col in fact_sales_clean.select_dtypes(include='object').columns:
    fact_sales_clean[col] = fact_sales_clean[col].astype(str).replace('nan', pd.NA).replace('NaT', pd.NA)

fact_sales_clean.to_parquet(DATA_PROCESSED / 'fact_sales_clean.parquet', index=False)

for col in product_enriched.select_dtypes(include='object').columns:
    product_enriched[col] = product_enriched[col].astype(str).replace('nan', pd.NA)
product_enriched.to_parquet(DATA_PROCESSED / 'products.parquet', index=False)

customer.to_parquet(DATA_PROCESSED / 'customers.parquet', index=False)
product_line.to_parquet(DATA_PROCESSED / 'product_lines.parquet', index=False)
product_group.to_parquet(DATA_PROCESSED / 'product_groups.parquet', index=False)
product_price.to_parquet(DATA_PROCESSED / 'product_prices.parquet', index=False)
province.to_parquet(DATA_PROCESSED / 'provinces.parquet', index=False)
sales_order.to_parquet(DATA_PROCESSED / 'sales_orders.parquet', index=False)

print('Saved to:', DATA_PROCESSED)
for f in sorted(DATA_PROCESSED.glob('*.parquet')):
    print(f'  {f.name} ({f.stat().st_size/1024:.0f} KB)')

Saved to: ../data/processed
  customers.parquet (42 KB)
  fact_sales_clean.parquet (627 KB)
  product_groups.parquet (3 KB)
  product_lines.parquet (4 KB)
  product_prices.parquet (15 KB)
  products.parquet (13 KB)
  provinces.parquet (4 KB)
  sales_orders.parquet (77 KB)
